# Theme 1: Bug-Fix Adoption Trends

**RQ1:** How does AI bug-fixing *volume* change over time?  
**RQ7:** Do developers *switch agents* for bug fixing over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)  
Scope: closed bug-fix PRs only (Copilot, Cursor, Claude Code, Devin + Human baseline)

**Methodology notes (applied to every figure below):**
- **Survivorship cutoff:** PRs created within the last 30 days of coverage are dropped — because we only see *closed* PRs, the most recent weeks are biased toward fast-decision PRs. `load_fix_prs` applies this automatically.
- **Model-version drift:** a single named agent spans several underlying models over 15 months, so within-agent trends mix product evolution with usage shift. Known model-release dates are annotated as grey vertical lines (see Claude Code).

> **v2 — AIDev repository sample.** This notebook re-runs the analysis restricted to the **true AIDev repository set** (1,743 repos from `hao-li/AIDev`, excl. Codex; 1,615 present in our data) via `load_fix_prs(restrict_to_repos=AIDEV_REPOS_TRUE)`. Figures are written to the `*_figures_v2/` folders. The broad-collection results (v1) are unchanged.

In [1]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Mahmoudabujadallah\CLI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    odds_ratio_ci, cliffs_delta, bh_correct,
    wilson_ci, bootstrap_median_ci, annotate_model_releases,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
    THEME1_DIR_V2, THEME2_DIR_V2, THEME3_DIR_V2, load_aidev_repo_set,
    MIN_N_PROP, MIN_N_MEDIAN,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [3]:
AIDEV_REPOS_TRUE = load_aidev_repo_set()
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs(restrict_to_repos=AIDEV_REPOS_TRUE)
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()
print('Agent fix PRs:', f"{len(agents_df):,}")
print('Human fix PRs:', f"{len(human_df):,}")

  AIDev repo set loaded: 1,743 repos
Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Restricted to 1,743 given repos: kept 305,739 of 371,577 PRs across 1,585 repos
  Fix PRs loaded: 305,739  |  Agent: 42,284  |  Human: 263,455


Agent fix PRs: 42,284
Human fix PRs: 263,455


## RQ1: How does AI bug-fixing volume change over time?

In [4]:
# Monthly volume: agent (all 4) vs human
monthly = df.groupby(['month', 'is_agent']).size().unstack(fill_value=0)
monthly.columns = ['Human', 'Agent']
monthly.index   = monthly.index.astype(str)
monthly['Total']    = monthly['Agent'] + monthly['Human']
monthly['Agent_%']  = (monthly['Agent'] / monthly['Total'] * 100).round(1)
print('Monthly volume (Agent vs Human):')
print(monthly.to_string())

Monthly volume (Agent vs Human):
         Human  Agent  Total  Agent_%
month                                
2024-12  14237   1893  16130     11.7
2025-01  16315   1928  18243     10.6
2025-02  17304   2132  19436     11.0
2025-03  19076   2558  21634     11.8
2025-04  19046   2548  21594     11.8
2025-05  18506   3036  21542     14.1
2025-06  18990   3913  22903     17.1
2025-07  21448   4860  26308     18.5
2025-08  18648   3464  22112     15.7
2025-09  20021   3276  23297     14.1
2025-10  20686   3140  23826     13.2
2025-11  19070   3149  22219     14.2
2025-12  19284   2984  22268     13.4
2026-01  20824   3403  24227     14.0


In [5]:
# Figure: monthly volume — agent vs human (line chart)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly.index, monthly['Agent'], 'o-', color='#1f77b4', label='Agent (all)')
ax.plot(monthly.index, monthly['Human'], 's--', color='#7f7f7f', label='Human', alpha=0.7)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff (Jul-25)')
ax.set_xlabel('Month')
ax.set_ylabel('Bug-Fix PRs')
ax.set_title('RQ1: Monthly Bug-Fix PR Volume — Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq1_volume_agent_vs_human', THEME1_DIR_V2)

  -> Saved: results\theme1_figures_v2\rq1_volume_agent_vs_human.png


WindowsPath('results/theme1_figures_v2/rq1_volume_agent_vs_human.png')

In [6]:
# Monthly volume per agent
per_agent = agents_df.groupby(['month', 'agent']).size().unstack(fill_value=0)
per_agent.index = per_agent.index.astype(str)
cols = [a for a in AGENTS if a in per_agent.columns]
per_agent = per_agent[cols]
print('Monthly volume by agent:')
print(per_agent.to_string())

Monthly volume by agent:
agent    Copilot  Cursor  Claude_Code  Devin
month                                       
2024-12       16    1012          796     69
2025-01       22    1061          746     99
2025-02       28    1090          871    143
2025-03       32    1240         1074    212
2025-04       35    1143         1160    210
2025-05      270    1465         1069    232
2025-06      599    1599         1549    166
2025-07     1064    1847         1804    145
2025-08      759    1483         1126     96
2025-09      578    1464         1163     71
2025-10      743    1280         1007    110
2025-11      904    1230          933     82
2025-12      728    1133          986    137
2026-01      986    1320          892    205


In [7]:
# Figure: per-agent stacked bar (monthly volume)
fig, ax = plt.subplots(figsize=(13, 5))
colors = [AGENT_COLORS[a] for a in per_agent.columns]
per_agent.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
ax.axvline(
    per_agent.index.tolist().index('2025-07') + 0.5,
    color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff'
)
ax.set_xlabel('Month')
ax.set_ylabel('Bug-Fix PRs')
ax.set_title('RQ1: Monthly Bug-Fix Volume by Agent')
ax.legend(loc='upper left')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq1_volume_by_agent', THEME1_DIR_V2)

  -> Saved: results\theme1_figures_v2\rq1_volume_by_agent.png


WindowsPath('results/theme1_figures_v2/rq1_volume_by_agent.png')

## RQ7: Do developers switch agents for bug fixing over time?

Measured as each agent's **share of total agent bug-fix PRs** per month.

In [8]:
# Agent market share: % of agent bug-fix PRs from each tool per month
share = per_agent.div(per_agent.sum(axis=1), axis=0) * 100
print('Agent market share (%) per month:')
print(share.round(1).to_string())

Agent market share (%) per month:
agent    Copilot  Cursor  Claude_Code  Devin
month                                       
2024-12      0.8    53.5         42.0    3.6
2025-01      1.1    55.0         38.7    5.1
2025-02      1.3    51.1         40.9    6.7
2025-03      1.3    48.5         42.0    8.3
2025-04      1.4    44.9         45.5    8.2
2025-05      8.9    48.3         35.2    7.6
2025-06     15.3    40.9         39.6    4.2
2025-07     21.9    38.0         37.1    3.0
2025-08     21.9    42.8         32.5    2.8
2025-09     17.6    44.7         35.5    2.2
2025-10     23.7    40.8         32.1    3.5
2025-11     28.7    39.1         29.6    2.6
2025-12     24.4    38.0         33.0    4.6
2026-01     29.0    38.8         26.2    6.0


In [9]:
# Figure: market share line chart
fig, ax = plt.subplots(figsize=(13, 5))
for agent in share.columns:
    ax.plot(share.index, share[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=2)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
# Annotate Claude Code model-release boundaries (within-agent product drift)
annotate_model_releases(ax, 'Claude_Code', list(share.index))
ax.set_xlabel('Month')
ax.set_ylabel('Share of Agent Bug-Fix PRs (%)')
ax.set_title('RQ7: Agent Market Share in Bug Fixing Over Time')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq7_agent_market_share', THEME1_DIR_V2)

  -> Saved: results\theme1_figures_v2\rq7_agent_market_share.png


WindowsPath('results/theme1_figures_v2/rq7_agent_market_share.png')

In [10]:
# Figure: stacked 100% bar — proportional agent usage
fig, ax = plt.subplots(figsize=(13, 5))
colors = [AGENT_COLORS[a] for a in share.columns]
share.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Share (%)')
ax.set_ylim(0, 100)
ax.set_title('RQ7: Proportional Agent Usage for Bug Fixing (Monthly)')
ax.legend(loc='upper right')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq7_agent_share_stacked', THEME1_DIR_V2)

  -> Saved: results\theme1_figures_v2\rq7_agent_share_stacked.png


WindowsPath('results/theme1_figures_v2/rq7_agent_share_stacked.png')

## Threats to validity (Theme 1)

- **Survivorship in the tail.** Only closed PRs are observed, so the most recent weeks over-represent fast-decision PRs. We drop the final 30 days, but months near the boundary remain partly affected — read the last 1–2 points cautiously.
- **Volume ≠ adoption.** PR counts reflect both how many developers use an agent *and* how active those repos are. A spike can be one very active repo (see the Jul-25 concentration check in `investigate.py`).
- **Market share is compositional.** RQ7 share is each agent's fraction of *agent* PRs; a rise for one agent mechanically lowers others. It does not measure absolute adoption.
- **Model-version drift.** "Claude_Code" (and every agent) spans multiple underlying models across the window; the annotated release lines mark where within-agent trends conflate product change with usage change.
- **Repo-coverage shift.** The set of repos in the dataset is not fixed month to month; volume/share trends partly track which repos entered coverage. RQ8 (Theme 3) addresses this by restricting to the AIDev repo set.